Build a Named Entity Recognition (NER) system for extracting entities from real-world text
such as news articles or social media data. And measure its accuracy, precision, recall, and F1-
score.

In [46]:
# !pip install spacy seqeval
# !python -m spacy download en_core_web_sm


In [ ]:
import spacy
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
import pandas as pd

In [48]:
nlp = spacy.load("en_core_web_sm")


In [49]:
# !pip install kagglehub

In [50]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("debasisdotcom/name-entity-recognition-ner-dataset")

# print("Path to dataset files:", path)

In [ ]:
path = "/home/kedar/.cache/kagglehub/datasets/debasisdotcom/name-entity-recognition-ner-dataset/versions/1"
df = pd.read_csv(
    f"{path}/NER dataset.csv",
    encoding="latin1",  # or "ISO-8859-1"
)

# /home/kedar/.cache/kagglehub/datasets/debasisdotcom/name-entity-recognition-ner-dataset/versions/1/NER dataset.csv
# print(path)

In [52]:
sentences = []
sentence = []

for _, row in df.iterrows():
    if isinstance(row["Sentence #"], str):  # new sentence
        if sentence:
            sentences.append(sentence)
        sentence = [(row["Word"], row["Tag"])]
    else:
        sentence.append((row["Word"], row["Tag"]))

if sentence:
    sentences.append(sentence)


In [ ]:
dataset = []

for sent in sentences:
    tokens = [w for w, t in sent]
    gold_tags = [t for w, t in sent]
    dataset.append({"tokens": tokens, "gold_tags": gold_tags})


In [54]:
# dataset = [
#     {
#         "tokens": ["Barack", "Obama", "was", "born", "in", "Hawaii"],
#         "gold_tags": ["B-PER", "I-PER", "O", "O", "O", "B-LOC"]
#     },
#     {
#         "tokens": ["Apple", "is", "based", "in", "California"],
#         "gold_tags": ["B-ORG", "O", "O", "O", "B-LOC"]
#     },
#     {
#         "tokens": ["Elon", "Musk", "founded", "SpaceX"],
#         "gold_tags": ["B-PER", "I-PER", "O", "B-ORG"]
#     },
#     {
#         "tokens": ["Google", "was", "founded", "by", "Larry", "Page"],
#         "gold_tags": ["B-ORG", "O", "O", "O", "B-PER", "I-PER"]
#     }
# ]


In [55]:
def predict_ner(tokens):
    doc = nlp(" ".join(tokens))
    pred_tags = ["O"] * len(tokens)

    for ent in doc.ents:
        for token in ent:
            if token.i < len(pred_tags):
                prefix = "B" if token.i == ent.start else "I"
                pred_tags[token.i] = f"{prefix}-{ent.label_}"

    return pred_tags


In [ ]:
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

y_true, y_pred = [], []

for sample in dataset[:500]:
    y_true.append(sample["gold_tags"])
    y_pred.append(predict_ner(sample["tokens"]))


In [57]:
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))


Precision: 0.0
Recall: 0.0
F1-score: 0.0


In [58]:
correct, total = 0, 0

for true_seq, pred_seq in zip(y_true, y_pred):
    for t, p in zip(true_seq, pred_seq):
        if t == p:
            correct += 1
        total += 1

accuracy = correct / total
print("Accuracy:", accuracy)


Accuracy: 0.7729591836734694


In [59]:
print(classification_report(y_true, y_pred))


              precision    recall  f1-score   support

    CARDINAL       0.00      0.00      0.00         0
        DATE       0.00      0.00      0.00         0
       EVENT       0.00      0.00      0.00         0
         FAC       0.00      0.00      0.00         0
         GPE       0.00      0.00      0.00         0
    LANGUAGE       0.00      0.00      0.00         0
         LOC       0.00      0.00      0.00         0
       MONEY       0.00      0.00      0.00         0
        NORP       0.00      0.00      0.00         0
     ORDINAL       0.00      0.00      0.00         0
         ORG       0.00      0.00      0.00         0
     PERCENT       0.00      0.00      0.00         0
      PERSON       0.00      0.00      0.00         0
     PRODUCT       0.00      0.00      0.00         0
    QUANTITY       0.00      0.00      0.00         0
        TIME       0.00      0.00      0.00         0
 WORK_OF_ART       0.00      0.00      0.00         0
         art       0.00    

/home/kedar/anaconda3/envs/nlp_env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/kedar/anaconda3/envs/nlp_env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
def extract_entities(text):
    doc = nlp(text)
    entities = []

    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities


In [62]:
text = "Apple is looking at buying U.K. startup for $1 billion"
entities = extract_entities(text)
print(entities)


[('Apple', 'ORG'), ('U.K.', 'GPE'), ('$1 billion', 'MONEY')]
